# Importacion de librerias

In [1]:
import pandas as pd
from dotenv import dotenv_values
import requests
from io import BytesIO
import unicodedata

#ignore warnings
import warnings
warnings.filterwarnings("ignore")

In [2]:
#Mostrar todas la columnas de los DataFrame
pd.set_option("display.max_columns", None)  # mostrar todas las columnas

# Utilizar los datos desde github

In [3]:
config = dotenv_values(dotenv_path=".env")

In [4]:
url = config['DATA_FILE_PATH']

headers = {"Authorization": f"Bearer {config['GITHUB_TOKEN']}"}
r = requests.get(url, headers=headers, timeout=30)
r.raise_for_status()  # Check if the request was successful

xlsx_data = pd.read_excel(BytesIO(r.content), header=None, sheet_name=None, engine="openpyxl")

# Funciones varias

In [5]:
# Limpiar cada hoja eliminando filas completamente vacías y restableciendo el índice
cleaned_sheets = {}
for name, sheet in xlsx_data.items():
    sheet = sheet.dropna(how="all")
    sheet = sheet.reset_index(drop=True)
    cleaned_sheets[name] = sheet

In [6]:
#Funcion para limpiar los nombres de las columnas y dejarlas mas estandarizadas
def clean_value(cadena):
  #Se eliminan espacios, se pasan a minusculas, se reemplazan espacios por guiones bajos, se eliminan acentos y caracteres especiales
  cadena = str(cadena).strip().lower().replace(' ', '_').replace('-','').replace("á", "a").replace("é", "e").replace("í", "i").replace("ó", "o").replace("ú", "u").replace(".","").replace("(","").replace(")","")
  return cadena

In [7]:
# funcion para detectar el formato de la tabla basándose en palabras clave en columnas específicas
def detect_table_format(sheet, keywords=['espacio'], check_columns=[0, 1]):
    """
    Detecta el formato de la tabla basándose en palabras clave en columnas específicas.
    
    Parámetros:
    - sheet: DataFrame a analizar
    - keywords: lista de palabras clave a buscar
    - check_columns: índices de columnas a verificar (0=primera columna, 1=segunda columna)
    
    Retorna:
    - 0 si la palabra clave se encuentra en la primera columna
    - 1 si la palabra clave se encuentra en la segunda columna
    - 'no determinada' si no se encuentra en ninguna
    """
    
    for col_idx in check_columns:
        if col_idx < len(sheet.columns):
            # Vectorizar la búsqueda: convierte a string y se limpia la columna de una sola vez
            col_data = sheet.iloc[:, col_idx].astype(str).str.lower().str.strip()
            
            # Aplicar transformaciones necesarias vectorizadas
            col_data_clean = col_data.str.replace(' ', '_', regex=False)
            col_data_clean = col_data_clean.str.replace('[áéíóú]', 
                lambda m: {'á':'a', 'é':'e', 'í':'i', 'ó':'o', 'ú':'u'}.get(m.group(0), m.group(0)), 
                regex=True)
            col_data_clean = col_data_clean.str.replace(r'[\-\.\(\)]', '', regex=True)
            
            # Verificar si alguna palabra clave existe en la columna
            pattern = '|'.join(keywords)
            if col_data_clean.str.contains(pattern, na=False, case=False).any():
                return col_idx
    
    return 'no determinada'

In [8]:
#Funcion para extraer la clasificación DC y edad del nombre de la hoja
def extract_patient_info(sheet_name):
    """Extrae la clasificación DC y edad del nombre de la hoja"""
    if 'F06' in str(sheet_name).upper():
        #Deterioro cognitivo leve
        dc = 1
    elif 'F02' in str(sheet_name).upper():
        #Demencia
        dc = 2
    elif 'GC' in str(sheet_name).upper():
        #Grupo de control
        dc = 0
    else: 
        dc = 'No determinada'
    
    age = str(sheet_name).split('-')[-1].strip()
    return dc, age

In [9]:
#funcion para extraer los valores de features de una tabla
def extract_features_from_table(sheet, headers_col, value_col, features, headers_clean=None):
    """
    Extrae los valores de features de una tabla.
    
    Parámetros:
    - sheet: DataFrame de la tabla
    - headers_col: índice de la columna con headers
    - value_col: índice de la columna con valores
    - features: lista de features a buscar
    - headers_clean: diccionario precominado de headers limpios
    
    Retorna: diccionario con los valores encontrados
    """
    # Pre-procesar headers para limpiarlos
    if headers_clean is None:
        headers = sheet.iloc[:, headers_col].astype(str)
        headers_clean = headers.apply(clean_value)
    else:
        headers_clean = headers_clean
    
    result = {}
    
    # Crear un diccionario para búsqueda rápida: feature_clean -> índice de fila
    feature_to_idx = {}
    for feature in features:
        mask = headers_clean.str.contains(feature, case=False, na=False)
        if mask.any():
            feature_to_idx[feature] = headers_clean[mask].index[0]
    
    # Extraer los valores para cada feature
    for feature in features:
        if feature in feature_to_idx:
            idx = feature_to_idx[feature]
            result[feature] = sheet.iloc[idx, value_col]
        else:
            result[feature] = None
    
    return result

In [10]:
# Función principal para buscar y extraer los valores de features de todas las tablas
def search_values(data, features):
    """
    Busca y extrae los valores de features de todas las tablas,
    diferenciando entre tabla formato 0 y formato 1.
    """
    # Configuración de parámetros para cada formato de tabla
    table_configs = {
        0: {'headers_col': 0, 'value_col': 3, 'nivel_estudio': 0},
        1: {'headers_col': 1, 'value_col': 4, 'nivel_estudio': 1}
    }
    
    results = []
    
    # Procesar todas las hojas
    for sheet_name, sheet in data.items():
        table_format = type_of_table[sheet_name]
        
        # Ignorar si el formato no se pudo determinar
        if table_format == 'no determinada':
            continue
        
        config = table_configs[table_format]
        
        # Extraer información del paciente
        dc, age = extract_patient_info(sheet_name)
        
        # Pre-procesar headers una sola vez para esta tabla
        headers = sheet.iloc[:, config['headers_col']].astype(str)
        headers_clean = headers.apply(clean_value)
        
        # Extraer features
        features_dict = extract_features_from_table(
            sheet, 
            config['headers_col'], 
            config['value_col'], 
            features,
            headers_clean
        )
        
        # Construir registro del paciente
        patient_data = {
            'sheet_name': sheet_name,
            'nivel_estudio': config['nivel_estudio'],
            'dc': dc,
            'age': age,
            **features_dict
        }
        
        results.append(patient_data)
    
    # Convertir a DataFrame
    dataset_final = pd.DataFrame(results)
    return dataset_final

# Lectura de datos

### Mapeo de columnas para los dos formatos de tablas

In [11]:
features = [
    
   #Tabla formato 0 -> Escolaridad baja
   #Tabla formato 1 -> Escolaridad alta
    
  #####Orientación
  
  #tabla 0 y tabla 1 con estos nombre ambos quedan en la misma columna
  'tiempo',
  'persona',
  'espacio', 
  
  #####Capacidad atencional

  #tabla 0
  'atencion_sostenida_auditiva', # -> 'digitos_en_progresion',
  'atencion_sostenida_visual', #Sin equivalencia
  'atencion_selectiva_visual', #Sin equivalencia
  'atencion_dividida_visual', # -> deteccion_visual
  #tabla 1
  'digitos_en_progresion',
#   'cubos_progresion',
  'deteccion_visual',
#   'deteccion_digitos_total',
  'series_sucesivas',

  #####Lenguaje
  
  #tabla 0
  'denominacion',
  'comprension_de_ordenes', #No crucial
  'material_verbal_complejo', #No crucial
  
  #tabla 1
  # 'denominacion'
  'semejanzas',
  # 'material_verbal_complejo',
  'comprension_de_ordenes',

  #####California verbal learning test | aprendizaje verbal del rey | memoria verbal
  
  #verbal del rey 
  
  #Tabla 0 - california verbal learning test
  'evocacion_inmediata_lista_a',
  'recuerdo_inmediato_lista_a',
  'recuerdo_inmediato_lista_b',
  'recuerdo_libre_a_corto_plazo',
  'recuerdo_libre_a_largo_plazo',
  'recuerdo_libre_a_largo_plazo',
  'reconocimiento',
  #tabla 0 - aprendizaje verbal del rey
  'evocacion_inmediata_lista_a',
  'recuerdo_inmediato_lista_b',
  'recuerdo_libre_a_corto_plazo',
  'recuerdo_libre_a_largo_plazo',
  'reconocimiento',
  #Tabla 1 #No tiene equivalencia con los otros test de la tabla 0
  'curva_de_memoria_volumen_promedio', 
  'memoria_verbal_espontanea_total',
  'memoria_verbal_claves_total',
  'memoria_verbal_reconocimiento',
  'memoria_lógica_promedio_historias',
  'memoria_lógica_promedio_historias',

  #####Memoria visual
  
  #Tabla 0
  'evocacion_diferida', # -> evocacion_figura_semicompleja
  #Tabla 1
  'caras_codificación',
  'reconocimiento_caras',
  'evocacion_figura_semicompleja',

  #####Genosias | capacidad visuoperceptiva
  #tabla 0
  'imagenes_sobrepuestas', # -> figuras_sobrepuestas
  'matrices',
  #tabla 1
  'imagenes_sobrepuestas',
  

  #####Praxis | praxias
  
  #Tabla 0
  'constructiva', #-> copia_de_figura_compleja
#   'ideomotora_gestos_simbolicos_a_la_orden_derecha',
#   'ideomotora_gestos_simbolicos_a_la_orden_izquierda',
#   'ideomotora_gestos_simbolicos_a_la_imitacion_derecha',
#   'ideomotora_gestos_simbolicos_a_la_imitacion_izquierda',
  #Tabla 1
  'copia_de_figura_compleja',
  'gestos_simbolicos',
  'imitacion_de_posturas',

  #####Funciones ejecutivas
  
  #Tabla 0
  'memoria_de_trabajo_digitos_inversos', # -> retencion_digitos_regresion
  'memoria_de_trabajo_digitos_secuenciales', # -> no tiene equivalencia
#   'test_de_fluidez_fonologica_fas_f',
#   'test_de_fluidez_fonologica_fas_a',
#   'test_de_fluidez_fonologica_fas_s',
  'fluidez_verbal_semantica',
  'semejanzas',
  'matrices',
  'comprension_abstraccion', # -> comprension_abstraccion
#   'atencion_alternante',
#   'stroop_a',
#   'stroop_b',
#   'stroop_c',
  'stroop_palabra',
  'stroop_color',
  'stroop_interferencia', # -> stroop_interferencia
  #tabla 1
  'semejanzas', # se puede presentar aqui o en lenguaje
  'fluidez_verbal_semantica',
  'fluidez_verbal_fonologica',
  'fluidez_no_verbal',
  'retencion_digitos_regresion'
  ]

### Mapeo de columas para la tabla con formato 0

In [12]:
features_tabla_0 = [
    
   #Tabla 0 nivel de estudio: Escolaridad baja
   #Tabla 1 nivel de estudio: Escolaridad alta
    
  #####Orientación
  
  #tabla 0 y tabla 1 con estos nombre ambos quedan en la misma columna
  'tiempo',
  'persona',
  'espacio', 
  
  #####Capacidad atencional

  #tabla 0
  'atencion_sostenida_auditiva', #'digitos_en_progresion',
  'atencion_sostenida_visual', #no tiene equivalencia
  'atencion_selectiva_visual', #no tiene equivalencia
  'atencion_dividida_visual', #Deteccion visual

  #####Lenguaje
  
  #tabla 0
  'denominacion',
  'material_verbal_complejo', #Es posible que no sea crucial
  'comprension_de_ordenes',

  #####California verbal learning test | aprendizaje verbal del rey | memoria verbal
  
  #verbal del rey 
  
  #Tabla 0 - california verbal learning test
  'evocacion_inmediata_lista_a',
  'recuerdo_inmediato_lista_a',
  'recuerdo_inmediato_lista_b',
  'recuerdo_libre_a_corto_plazo',
  'recuerdo_libre_a_largo_plazo',
  'recuerdo_libre_a_largo_plazo',
  'reconocimiento',
  #tabla 0 - aprendizaje verbal del rey
  'evocacion_inmediata_lista_a',
  'recuerdo_inmediato_lista_b',
  'recuerdo_libre_a_corto_plazo',
  'recuerdo_libre_a_largo_plazo',
  'reconocimiento',


  #####Memoria visual
  
  #Tabla 0
  'evocacion_diferida',#evocacion_figura_semicompleja

  #####Genosias | capacidad visuoperceptiva
  
  #tabla 0
  'imagenes_sobrepuestas',
  'matrices',

  #####Praxis | praxias
  
  #Tabla 0
  'constructiva', #copia_de_figura_compleja
#   'ideomotora_gestos_simbolicos_a_la_orden_derecha',
#   'ideomotora_gestos_simbolicos_a_la_orden_izquierda',
#   'ideomotora_gestos_simbolicos_a_la_imitacion_derecha',
#   'ideomotora_gestos_simbolicos_a_la_imitacion_izquierda',

  #####Funciones ejecutivas
  
  #Tabla 0
  'memoria_de_trabajo_digitos_inversos', #retencion_digitos_regresion
  'memoria_de_trabajo_digitos_secuenciales', #no tiene equivalencia
#   'test_de_fluidez_fonologica_fas_f',
#   'test_de_fluidez_fonologica_fas_a',
#   'test_de_fluidez_fonologica_fas_s',
  'fluidez_verbal_semantica',
  'semejanzas',
  'matrices',
  'comprension_abstraccion', #comprension_abstraccion
#   'atencion_alternante',
#   'stroop_a',
#   'stroop_b',
#   'stroop_c',
  'stroop_palabra',
  'stroop_color',
  'stroop_interferencia', #stroop_interferencia
  ]

### Mapeo de columnas para la tabla con formato 1

In [13]:
features_tabla_1 = [
    
   #Tabla 0 nivel de estudio: Escolaridad baja
   #Tabla 1 nivel de estudio: Escolaridad alta
    
  #####Orientación
  
  #tabla 0 y tabla 1 con estos nombre ambos quedan en la misma columna
  'tiempo',
  'persona',
  'espacio', 
  
  #####Capacidad atencional

  #tabla 1
  'digitos_en_progresion',
#   'cubos_progresion',
  'deteccion_visual',
#   'deteccion_digitos_total',
  'series_sucesivas',

  #####Lenguaje
  

  # #tabla 1
  'denominacion',
  'semejanzas',
  'material_verbal_complejo',
  'comprension_de_ordenes',

  #####California verbal learning test | aprendizaje verbal del rey | memoria verbal
  
  #verbal del rey 
  
  #Tabla 1 #No tiene equivalencia con los otros test de la tabla 0
  'curva_de_memoria_volumen_promedio', 
  'memoria_verbal_espontanea_total',
  'memoria_verbal_claves_total',
  'memoria_verbal_reconocimiento',
  'memoria_logica_promedio_historias',

  #####Memoria visual
  
  #Tabla 1
  'caras_codificacion',
  'reconocimiento_caras',
  'evocacion_figura_semicompleja',

  #####Genosias | capacidad visuoperceptiva
  
  # tabla 1
  'imagenes_sobrepuestas',

  #####Praxis | praxias
  
  #Tabla 1
  'copia_de_figura_compleja',
  'gestos_simbolicos',

  #####Funciones ejecutivas

  #tabla 1
  'semejanzas', # se puede presentar aqui o en lenguaje
  'fluidez_verbal_semantica',
  'fluidez_verbal_fonologica',
  'fluidez_no_verbal',
  'retencion_digitos_regresion',
  ]

### Mapeo de columnas con las dos tablas en conjunto

In [14]:
features_tablas = [
    
   #Tabla 0 nivel de estudio: Escolaridad baja
   #Tabla 1 nivel de estudio: Escolaridad alta
    
  #####Orientación
  
  #tabla 0 y tabla 1 con estos nombre ambos quedan en la misma columna
  'tiempo',
  'persona',
  'espacio', 
  
  #####Capacidad atencional

  #tabla 0
  'atencion_sostenida_auditiva', #'digitos_en_progresion',
  'atencion_dividida_visual', #deteccion_visual
  #tabla 1
  'digitos_en_progresion',
  'deteccion_visual',

  #####Lenguaje
  
  #tabla 0
  'denominacion',
  'material_verbal_complejo',
  'semejanzas',
  # #tabla 1
  # 'denominacion'
#   'semejanzas'

  #####California verbal learning test | aprendizaje verbal del rey | memoria verbal
  
  #####Memoria visual
  
  #Tabla 0
  'evocacion_diferida',#evocacion_figura_semicompleja
  #Tabla 1
  'evocacion_figura_semicompleja',

  #####Genosias | capacidad visuoperceptiva
  # Las genosias se quedarian sin analizar

  #####Praxis | praxias
  
  #Tabla 0
  'constructiva', #copia_de_figura_compleja
  #Tabla 1
  'copia_de_figura_compleja',

  #####Funciones ejecutivas
  
  #Tabla 0
  'memoria_de_trabajo_digitos_inversos', #retencion_digitos_regresion
  'comprension_abstraccion', #comprension_abstraccion
  'comprension',
  #tabla 1
  'retencion_digitos_regresion',
  ]

### Hojas procesadas

In [15]:
print(f'Cantidad de datos {len(cleaned_sheets)} \n')
print(f'Nombre de las hojas procesadas:\n ')
sheets = list()
for name, sheet in cleaned_sheets.items():
    sheets.append(name)
print(sheets)

Cantidad de datos 125 

Nombre de las hojas procesadas:
 
['S1-F067-58', 'GC1-60', 'GC2-62', 'F021-72', 'S2-F067-53', 'GC3-60', 'S3-F067-66', 'GC4-72', 'GC5-62', 'S4-F067-66', 'GC6-69', 'GC7-70', 'F022-71', 'GC8-79', 'S5-F067-73', 'F023-87', 'F024-72', 'S6-F067-65', 'F025-82', 'GC8-80', 'GC9-66', 'GC10-65', 'F026-92', 'S7-F067-76', 'S8-F067-75', 'F027-79', 'GC11-72', 'S9-F067-66', 'S10-F067-65', 'GC12-65', 'S11-F067-84', 'F028-70', 's12-f067-77', 'S13-F067-77', 'F029-62', 'S14-F067-69', 'F0210-77', 'F0211-79', 'S15-F067-74', 'F0212-65', 'GC13-74', 'S16-F067-60', 'GC14-67', 'F0213-85', 'S17-F067-76', 'S18-F067-72', 'F0214-73', 'S19-F064-74', 'S20-F067-57', 'S21-F067-68', 'S22-F067-86', 'GC15-64', 'S23-F067-66', 'S24-F067-96', 'GC16-76', 'F0215-73', 'S25-F067-80', 'F0216-83', 'F0217-88', 'F0218-77', 'S26-F067-81', 'S27-F067-66', 'S28-F067-65', 'S29-F067-82', 'F0219-74', 'S30-F067-87', 'S31-F067-83', 'S32-F067-60', 'S33-F067-57', 'F0220-77', 'F0221-57', 'S34-F067-71', 'F0222-64', 'S35-F06

### Formatos de tablas

In [16]:
print('Formato de tabla 1:')
cleaned_sheets['S1-F067-58']

Formato de tabla 1:


,0,1,2,3,4,5,6,7,8,9,10,11
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Alto: 14 a 19 puntos,NaN,Alto: 80 a 95 puntos
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Promedio: 7 a13 puntos,NaN,Promedio: 30 a 70 puntos
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Alteración leve- moderado: 4 a 6 puntos,NaN,Bajo: 10 a 20 puntos
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Alteración severa: 1 a 3 puntos,NaN,Déficit: <5 puntos
4,Dominio-Neuropsi Atención y Memoria- Test de B...,NaN,Puntuación obtenida,Puntuación normalizada,Interpretación,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,Orientación,Tiempo,2/4 p,1,Alteración severa,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,Espacio,1/2 p,1,Alteración severa,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,NaN,Persona,1/1 p,11,Promedio,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,Atención y concentración,Dígitos en progresión,3/9 p,3,Alteración severa,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,NaN,Detección visual total *2 comisiones*,9,6,Alteración Leve,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
print('Formato de tabla 0:')
cleaned_sheets['GC4-72']

Formato de tabla 0:


,0,1,2,3
0,FUNCIÓN EVALUADA (TEST EMPLEADO),P. Directa,Percentil o P. Directa,Clasificación
1,Orientación,NaN,NaN,NaN
2,Orientación en persona (TB),7 de 7,95,Alto
3,Orientación en espacio (TB),5 de 5,95,Alto
4,Orientación en tiempo (TB),20 de 23,5,Bajo
5,Capacidad atencional y velocidad de procesamiento,NaN,NaN,NaN
6,Atención sostenida auditiva (Dígitos directos ...,6/16 p,8,Promedio
7,Atención sostenida visual (TMT-A),32 sg,95,Alto
8,Atención selectiva visual (Búsqueda de símbolos),29,13,Promedio
9,Atención dividida visual (Clave de números-WAIS),42,11,Promedio


In [18]:
# Aplicar la detección a todas las hojas de manera vectorizada
type_of_table = {name: detect_table_format(sheet) for name, sheet in cleaned_sheets.items()}
print(f'Cantidad de datos formato tabla 0: {list(type_of_table.values()).count(0)}')
print(f'Cantidad de datos formato tabla 1: {list(type_of_table.values()).count(1)}')
# print(f'Cantidad de datos con formato no determinado: {list(type_of_table.values()).count("no determinada")}')
print(f'Cantidad total de datos procesados: {len(type_of_table)}')

Cantidad de datos formato tabla 0: 36
Cantidad de datos formato tabla 1: 81
Cantidad total de datos procesados: 125


# Creación de los datasets

In [19]:
# Los dos formatos de tablas en conjunto
df_tablas = search_values(cleaned_sheets, features_tablas)

# Formato de tabla 0
cleaned_sheets_tabla_0 = {name: sheet for name, sheet in cleaned_sheets.items() if type_of_table[name] == 0}
df_tabla_0 = search_values(cleaned_sheets_tabla_0, features_tabla_0)

# Formato de tabla 1
cleaned_sheets_tabla_1 = {name: sheet for name, sheet in cleaned_sheets.items() if type_of_table[name] == 1}
df_tabla_1 = search_values(cleaned_sheets_tabla_1, features_tabla_1)

In [20]:
print(f'Grupo de control: {len(df_tablas[df_tablas["dc"] == 0])} personas')
print(f'Grupo de deterioro cognitivo: {len(df_tablas[df_tablas["dc"] == 1])} personas')
print(f'Grupo con demencia: {len(df_tablas[df_tablas["dc"] == 2])} personas')
print(f'Cantidad total de personas en el dataset: {len(df_tablas)} personas')

Grupo de control: 27 personas
Grupo de deterioro cognitivo: 58 personas
Grupo con demencia: 32 personas
Cantidad total de personas en el dataset: 117 personas


### Samples

In [21]:
df_tabla_1.sample(5)

,sheet_name,nivel_estudio,dc,age,tiempo,persona,espacio,digitos_en_progresion,deteccion_visual,series_sucesivas,denominacion,semejanzas,material_verbal_complejo,comprension_de_ordenes,curva_de_memoria_volumen_promedio,memoria_verbal_espontanea_total,memoria_verbal_claves_total,memoria_verbal_reconocimiento,memoria_logica_promedio_historias,caras_codificacion,reconocimiento_caras,evocacion_figura_semicompleja,imagenes_sobrepuestas,copia_de_figura_compleja,gestos_simbolicos,fluidez_verbal_semantica,fluidez_verbal_fonologica,fluidez_no_verbal,retencion_digitos_regresion
25,F0211-79,1,2,79,Alteración severa,Promedio,Promedio,Promedio,Alteración leve,Promedio,Alto,Promedio,Alto,Alto,Promedio,Promedio,Alteración leve,Promedio,Alteración severa,Promedio,Promedio,None,Alto,Promedio,Alto,Alteración leve,Promedio,Alteración severa,Alteración leve
63,GC19-53,1,0,53,Promedio,Promedio,Promedio,Promedio,Promedio,Promedio,Alto,Promedio,Alto,Alto,Promedio,Promedio,Bajo,Promedio,None,Promedio,Promedio,Promedio,Alto,Promedio,Alto,Promedio,Promedio,Promedio,Promedio
8,GC9-66,1,0,66,Promedio,Promedio,Promedio,Promedio,Promedio,Promedio,Alto,Alto,Déficit,None,Promedio,Promedio,Alteración Leve,Promedio,Promedio,Promedio,Promedio,Promedio,Alto,None,None,Promedio,Promedio,Promedio,Promedio
3,F023-87,1,2,87,Alteración severa,Alteración severa,Alteración severa,Promedio,Alteración severa,Promedio,Deficit,Deficit,Déficit,None,Promedio,Alteración leve,Alteración leve,Alteración leve,Alteración leve,Promedio,Promedio,Alteración leve,Déficit,None,Alto,Alteración leve,Alteración severa,Alteración leve,Alteración severa
0,S1-F067-58,1,1,58,Alteración severa,Promedio,Alteración severa,Alteración severa,Alteración Leve,Promedio,Alto,Promedio,Alto,Alteración Leve,Promedio,Promedio,Alteración Leve,Promedio,Promedio,Promedio,Promedio,Promedio,Alto,Promedio,Alto,Alteración Leve,Alteración Leve,Promedio,Alteración Leve


In [22]:
df_tabla_0.sample(5)

,sheet_name,nivel_estudio,dc,age,tiempo,persona,espacio,atencion_sostenida_auditiva,atencion_sostenida_visual,atencion_selectiva_visual,atencion_dividida_visual,denominacion,material_verbal_complejo,comprension_de_ordenes,evocacion_inmediata_lista_a,recuerdo_inmediato_lista_a,recuerdo_inmediato_lista_b,recuerdo_libre_a_corto_plazo,recuerdo_libre_a_largo_plazo,reconocimiento,evocacion_diferida,imagenes_sobrepuestas,matrices,constructiva,memoria_de_trabajo_digitos_inversos,memoria_de_trabajo_digitos_secuenciales,fluidez_verbal_semantica,semejanzas,comprension_abstraccion,stroop_palabra,stroop_color,stroop_interferencia
21,GC21-64,0,0,64,Alto,Alto,Alto,Promedio,Promedio**,Promedio**,Promedio,Alto,None,Alto,Promedio,Promedio,Promedio,Promedio,Promedio,Promedio,Promedio,None,None,None,Promedio,Promedio,None,Promedio,Promedio,Promedio,Promedio,Bajo
29,S53-F067-54,0,1,54,Alto,Alto,Alto,Bajo,Promedio,Bajo,Bajo,None,Alto,Alto,Bajo,Bajo,Bajo,Bajo,Bajo,Bajo,Promedio,None,Bajo,None,Bajo,Bajo,Bajo,Bajo,Déficit,None,None,None
2,F021-72,0,2,72,Alto,Alto,Alto,Promedio,Déficit,Promedio,Promedio,Alto,Alto,Alto,Bajo,Bajo,Bajo,Bajo,Bajo,Promedio,Bajo,Alto,None,Promedio,Bajo,Promedio,Déficit,Alto,Déficit,Déficit,Déficit,Déficit
18,GC17-60,0,0,60,Alto,Alto,Alto,Promedio,Alto,Promedio,Promedio,Alto,Bajo,Alto,Promedio,Bajo,Promedio,Bajo,Bajo,Promedio**,Promedio,Alto,Promedio,None,Bajo,Promedio,None,Promedio,Déficit,Promedio,Promedio,Promedio
30,S54-F067-68,0,1,68,Alto,Alto,Alto,Promedio,Promedio,Promedio,Promedio,Alto,Alto,Alto,Bajo,Bajo,Bajo,Bajo,Bajo,Bajo,Bajo,Alto,Promedio,None,Alto,Promedio,Promedio,Bajo,Promedio,Promedio,Bajo,None


In [23]:
df_tablas.sample(5)

,sheet_name,nivel_estudio,dc,age,tiempo,persona,espacio,atencion_sostenida_auditiva,atencion_dividida_visual,digitos_en_progresion,deteccion_visual,denominacion,material_verbal_complejo,semejanzas,evocacion_diferida,evocacion_figura_semicompleja,constructiva,copia_de_figura_compleja,memoria_de_trabajo_digitos_inversos,comprension_abstraccion,comprension,retencion_digitos_regresion
59,F0218-77,1,2,77,Alteración severa,Promedio,Promedio,None,None,Promedio,Promedio,Alto,Alto,Promedio,None,Alteración leve,None,Alteración severa,None,None,Promedio,Alteración leve
86,S42-F067-68,1,1,68,Alteracion Severa,Alteracion Severa,Alteracion Severa,None,None,Promedio,Alteración Leve,Alteración Leve,Alteración Leve,Déficit,None,Alteración Leve,Alto,Alteracion Severa,None,Déficit,Alteración Leve,Alteración Leve
106,F0230-67,1,2,67,Déficit,Promedio,Promedio,None,None,Promedio,Promedio,Máximo,Promedio,Déficit,None,Déficit,Déficit,None,None,None,None,Bajo
35,S14-F067-69,1,1,69,Promedio,Promedio,Promedio,None,None,Promedio,Alteración Leve,Alto,Bajo,Promedio,None,None,None,Alto,None,None,Alto,Promedio
89,F0228-86,1,2,86,Alteración Severa,Alteración Severa,Alteración Severa,None,None,Promedio,Alteración Severa,Alto,Deficit,Promedio,None,Alteración Leve,Bajo,Alteración Severa,None,None,Deficit,Alteración Leve


### Corrección de nombre y rellenado de nulos

In [24]:
df_tablas['atencion_sostenida_auditiva'] = df_tablas['atencion_sostenida_auditiva'].fillna(df_tablas['digitos_en_progresion'])
df_tablas['atencion_dividida_visual'] = df_tablas['atencion_dividida_visual'].fillna(df_tablas['deteccion_visual'])
df_tablas['evocacion_diferida'] = df_tablas['evocacion_diferida'].fillna(df_tablas['evocacion_figura_semicompleja'])
df_tablas['constructiva'] = df_tablas['constructiva'].fillna(df_tablas['copia_de_figura_compleja'])
df_tablas['memoria_de_trabajo_digitos_inversos'] = df_tablas['memoria_de_trabajo_digitos_inversos'].fillna(df_tablas['retencion_digitos_regresion'])
df_tablas['comprension_abstraccion'] = df_tablas['comprension_abstraccion'].fillna(df_tablas['comprension'])



df_tablas.drop(columns=['digitos_en_progresion'], inplace=True)
df_tablas.drop(columns=['deteccion_visual'], inplace=True)
df_tablas.drop(columns=['evocacion_figura_semicompleja'], inplace=True)
df_tablas.drop(columns=['copia_de_figura_compleja'], inplace=True)
df_tablas.drop(columns=['retencion_digitos_regresion'], inplace=True)
df_tablas.drop(columns=['comprension'], inplace=True)

# atencion_sostenida_auditiva->digitos_en_progresion o
# atencion_dividida_visual->deteccion_visual o
# evocacion_diferida->evocacion_figura_semicompleja o
# constructiva->copia_de_figura_compleja o
# memoria_de_trabajo_digitos_inversos->retencion_digitos_regresion o

# Normalización de datos

In [25]:
# Función para limpiar tildes
def limpiar_tildes(texto):
    texto_nfd = unicodedata.normalize('NFD', str(texto))
    return ''.join(c for c in texto_nfd if unicodedata.category(c) != 'Mn')

def limpiar_caracteres_especiales(texto):
    # Eliminar caracteres especiales excepto espacios y guiones bajos y guiones
    return ''.join(c for c in str(texto) if c.isalnum() or c in [' ', '_', '-'])

def limpiar_texto(texto):
    if pd.isna(texto) or texto == 'None' or texto == 'nan':
        return None
    
    # LUEGO aplicar las limpiezas
    texto = str(texto)
    texto = limpiar_tildes(texto)
    texto = limpiar_caracteres_especiales(texto)
    texto = texto.lower().strip()
    
    return texto if texto else None

### Tabla 0

In [26]:
#Alteracion severa - 1,2,3
#bajo - 4,5,6
#promedio - 7-13
#Alto - 14-18

In [27]:
# Reemplazo de valores
for col in df_tabla_1.columns:
    # Aplicar limpieza de tildes antes de hacer los reemplazos
    df_tabla_1[col] = df_tabla_1[col].apply(limpiar_texto)
    df_tabla_1[col] = df_tabla_1[col].replace('alteracion severa', 'alteracion_severa')
    df_tabla_1[col] = df_tabla_1[col].replace('altearcion severa', 'alteracion_severa')
    df_tabla_1[col] = df_tabla_1[col].replace('alteracion leve', 'bajo')
    df_tabla_1[col] = df_tabla_1[col].replace('alteracion', 'bajo')
    df_tabla_1[col] = df_tabla_1[col].replace('alteracion moderada', 'bajo')
    df_tabla_1[col] = df_tabla_1[col].replace('aleracion leve', 'bajo')
    df_tabla_1[col] = df_tabla_1[col].replace('deficit', 'bajo')
    df_tabla_1[col] = df_tabla_1[col].replace('alteracion leve-moderada', 'bajo')
    df_tabla_1[col] = df_tabla_1[col].replace('alteracion leve-moderada', 'bajo')
    df_tabla_1[col] = df_tabla_1[col].replace('alto', 'alto')
    df_tabla_1[col] = df_tabla_1[col].replace('promedio', 'promedio')
    df_tabla_1[col] = df_tabla_1[col].replace('normal alto', 'alto')
    df_tabla_1[col] = df_tabla_1[col].replace('maximo', 'alto')

In [28]:
#mostrar los posibles valores de todas las columnas menos dc, age, nivel_estudio, sheet_name

cols = df_tabla_1.columns.difference(['dc', 'age', 'nivel_estudio', 'sheet_name'])
for col in cols:
    print(f'Valores únicos en la columna "{col}": {df_tabla_1[col].unique()}')

Valores únicos en la columna "caras_codificacion": ['promedio' None 'alto' 'alteracion_severa']
Valores únicos en la columna "comprension_de_ordenes": ['bajo' None 'alto' 'promedio']
Valores únicos en la columna "copia_de_figura_compleja": ['promedio' None 'bajo' 'alto' 'alteracion_severa']
Valores únicos en la columna "curva_de_memoria_volumen_promedio": ['promedio' 'bajo' 'alto' 'alteracion_severa' None]
Valores únicos en la columna "denominacion": ['alto' 'bajo' 'promedio' None]
Valores únicos en la columna "deteccion_visual": ['bajo' 'promedio' 'alteracion_severa' 'alto' None]
Valores únicos en la columna "digitos_en_progresion": ['alteracion_severa' 'promedio' 'bajo' None 'alto']
Valores únicos en la columna "espacio": ['alteracion_severa' 'promedio' None]
Valores únicos en la columna "evocacion_figura_semicompleja": ['promedio' 'bajo' 'alteracion_severa' None 'alto']
Valores únicos en la columna "fluidez_no_verbal": ['promedio' 'bajo' 'alteracion_severa' 'alto' None]
Valores únic

### Tabla 1

In [29]:
#tabla 1
# menor a 5 alteracion Alteración Severa
# 5 a 24 bajo   
# 25 a 85 promedio
# 85 a 100 Alto

In [30]:
# Reemplazo de valores
for col in df_tabla_0.columns:
    # Aplicar limpieza de tildes antes de hacer los reemplazos
    df_tabla_0[col] = df_tabla_0[col].apply(limpiar_texto)
    df_tabla_0[col] = df_tabla_0[col].replace('alto', 'alto')
    df_tabla_0[col] = df_tabla_0[col].replace('deficit', 'alteracion_severa')
    df_tabla_0[col] = df_tabla_0[col].replace('promedio', 'promedio')
    df_tabla_0[col] = df_tabla_0[col].replace('bajo', 'bajo')
    df_tabla_0[col] = df_tabla_0[col].replace('maximo', 'alto')


In [31]:
#mostrar los posibles valores de todas las columnas menos dc, age, nivel_estudio, sheet_name

cols = df_tabla_0.columns.difference(['dc', 'age', 'nivel_estudio', 'sheet_name'])
for col in cols:
    print(f'Valores únicos en la columna "{col}": {df_tabla_0[col].unique()}')

Valores únicos en la columna "atencion_dividida_visual": [None 'promedio' 'bajo']
Valores únicos en la columna "atencion_selectiva_visual": [None 'promedio' 'bajo']
Valores únicos en la columna "atencion_sostenida_auditiva": [None 'promedio' 'bajo' 'alto']
Valores únicos en la columna "atencion_sostenida_visual": [None 'promedio' 'alteracion_severa' 'bajo' 'alto']
Valores únicos en la columna "comprension_abstraccion": [None 'promedio' 'alteracion_severa' 'alto' 'bajo']
Valores únicos en la columna "comprension_de_ordenes": [None 'alto' 'bajo' 'alteracion_severa']
Valores únicos en la columna "constructiva": [None 'promedio']
Valores únicos en la columna "denominacion": [None 'alto']
Valores únicos en la columna "espacio": [None 'alto' 'bajo']
Valores únicos en la columna "evocacion_diferida": [None 'promedio' 'bajo' 'alto' 'alteracion_severa']
Valores únicos en la columna "evocacion_inmediata_lista_a": [None 'promedio' 'bajo']
Valores únicos en la columna "fluidez_verbal_semantica": [

# Verificación de nulos

In [32]:
print(df_tabla_0.shape)
df_tabla_0.isna().sum()

(36, 32)


,0
sheet_name,0
nivel_estudio,0
dc,0
age,0
tiempo,1
persona,1
espacio,1
atencion_sostenida_auditiva,1
atencion_sostenida_visual,4
atencion_selectiva_visual,1


In [33]:
print(df_tabla_1.shape)
df_tabla_1.isna().sum()

(81, 29)


,0
sheet_name,0
nivel_estudio,0
dc,0
age,0
tiempo,3
persona,3
espacio,3
digitos_en_progresion,3
deteccion_visual,6
series_sucesivas,3


# Manejo de valores nulos

En datasets clinicos, los nulos no se tratan solo como ruido estadistico. Pueden reflejar fatiga del paciente, severidad del cuadro, o ausencia de una prueba por criterio clinico. Por eso el manejo debe:

1. **Conservar pacientes** (evitar eliminar filas).
2. **Modelar la ausencia** (indicadores de nulo por variable).
3. **Imputar por grupo diagnostico** (`dc`) para no borrar diferencias clinicas.
4. **Evitar imputar variables con demasiada ausencia** (mantener nulo e indicador).

Criterios propuestos:
- **< 5% nulos**: imputacion simple por mediana (numerico) o moda (categorico).
- **5-30% nulos**: imputacion por grupo `dc` + indicador de ausencia.
- **>= 30% nulos**: mantener nulo + indicador, revisar clinicamente.

El objetivo es conservar informacion clinica y minimizar sesgos.

In [34]:
# Funciones para perfilado e imputacion clinica de nulos
import numpy as np

ID_COLS = ['dc', 'age', 'nivel_estudio', 'sheet_name']


def null_data_info(df, id_cols):
    cols = [c for c in df.columns if c not in id_cols] #Se omiten las columnas de identificación para el perfilado de nulos
    resumen = pd.DataFrame({
        'nulos': df[cols].isna().sum(),
        '%_nulos': (df[cols].isna().mean() * 100).round(2),
        'dtype': df[cols].dtypes.astype(str),
        'n_unicos': df[cols].nunique(dropna=True)
    }).sort_values('%_nulos', ascending=False)
    return resumen


def imputacion_null(df, id_cols, group_col='dc', high_missing=0.30):
    df_imp = df.copy()
    cols = [col for col in df.columns if col not in id_cols]
    missing_rate = df[cols].isna().mean() #El porcentaje de nulos por columna se calcula como el número de nulos dividido por el total de filas, lo que da un valor entre 0 y 1 representando el porcentaje de nulos en cada columna.

    for col in cols:
        porcent = missing_rate[col]
        if porcent == 0:
            continue

        # Indicador de ausencia
        df_imp[f'{col}_missing'] = df[col].isna().astype(int)

        # Si hay demasiados nulos, no imputar
        if porcent >= high_missing:
            continue

        # Imputacion por tipo de dato
        if df[col].dtype == 'object': #Si la columna es de tipo object, se asume que es categórica y se imputa con la moda por grupo clínico. La función fill_mode calcula la moda de cada grupo definido por group_col (en este caso, 'dc') y luego llena los valores nulos con esa moda. Si no hay una moda clara (es decir, si hay un empate), se toma el primer valor de la moda. Si aún quedan nulos después de esta imputación, se realiza una imputación global utilizando la moda de toda la columna.
            # Moda por grupo clinico
            def fill_mode(s):
                #Se calcula la moda de la serie s, omitiendo los valores nulos. Si hay una moda clara, se toma el primer valor; si no, se asigna NaN. Luego, se llena la serie s con este valor de moda para los valores nulos.
                mode = s.mode(dropna=True)
                fill_value = mode.iloc[0] if len(mode) > 0 else np.nan
                return s.fillna(fill_value)

            df_imp[col] = df.groupby(group_col)[col].transform(fill_mode)

            # Fallback global si aun quedan nulos despues de la imputacion por grupo clinico
            if df_imp[col].isna().any():
                global_mode = df[col].mode(dropna=True)
                if len(global_mode) > 0:
                    df_imp[col] = df_imp[col].fillna(global_mode.iloc[0])
        else:
            # Si la variable no es categórica, se asume que es numérica y se imputa con la mediana por grupo clínico. La función fill_median calcula la mediana de cada grupo definido por group_col (en este caso, 'dc') y luego llena los valores nulos con esa mediana. Si aún quedan nulos después de esta imputación, se realiza una imputación global utilizando la mediana de toda la columna.
            df_imp[col] = df.groupby(group_col)[col].transform(lambda s: s.fillna(s.median()))

            # Fallback global si aun quedan nulos
            if df_imp[col].isna().any():
                df_imp[col] = df_imp[col].fillna(df[col].median())

    return df_imp


# Perfil de nulos
print('=== PERFIL NULOS TABLA 0 ===')
perfil_t0 = null_data_info(df_tabla_0, ID_COLS)
print(perfil_t0[perfil_t0['nulos'] > 0])

print('\n=== PERFIL NULOS TABLA 1 ===')
perfil_t1 = null_data_info(df_tabla_1, ID_COLS)
print(perfil_t1[perfil_t1['nulos'] > 0])

# Imputacion clinica (sin eliminar pacientes)
df_tabla_0_imp = imputacion_null(df_tabla_0, ID_COLS)
df_tabla_1_imp = imputacion_null(df_tabla_1, ID_COLS)

print('\nTablas imputadas:')
print('df_tabla_0_imp ->', df_tabla_0_imp.shape)
print('df_tabla_1_imp ->', df_tabla_1_imp.shape)


=== PERFIL NULOS TABLA 0 ===
                                         nulos  %_nulos   dtype  n_unicos
constructiva                                35    97.22  object         1
fluidez_verbal_semantica                    23    63.89  object         4
matrices                                    12    33.33  object         4
imagenes_sobrepuestas                       11    30.56  object         3
stroop_interferencia                         9    25.00  object         4
material_verbal_complejo                     6    16.67  object         3
denominacion                                 5    13.89  object         1
stroop_palabra                               4    11.11  object         4
evocacion_diferida                           4    11.11  object         4
stroop_color                                 4    11.11  object         4
atencion_sostenida_visual                    4    11.11  object         4
memoria_de_trabajo_digitos_secuenciales      3     8.33  object         2
comprensi

In [35]:
df_tabla_0[df_tabla_0['sheet_name'] == 's3-f067-66']

,sheet_name,nivel_estudio,dc,age,tiempo,persona,espacio,atencion_sostenida_auditiva,atencion_sostenida_visual,atencion_selectiva_visual,atencion_dividida_visual,denominacion,material_verbal_complejo,comprension_de_ordenes,evocacion_inmediata_lista_a,recuerdo_inmediato_lista_a,recuerdo_inmediato_lista_b,recuerdo_libre_a_corto_plazo,recuerdo_libre_a_largo_plazo,reconocimiento,evocacion_diferida,imagenes_sobrepuestas,matrices,constructiva,memoria_de_trabajo_digitos_inversos,memoria_de_trabajo_digitos_secuenciales,fluidez_verbal_semantica,semejanzas,comprension_abstraccion,stroop_palabra,stroop_color,stroop_interferencia
5,s3-f067-66,0,1,66,alto,alto,alto,promedio,bajo,promedio,bajo,alto,bajo,alto,bajo,bajo,promedio,bajo,bajo,bajo,bajo,alto,None,None,promedio,None,promedio,promedio,promedio,bajo,bajo,None


# Creación de tabla conjunta

In [36]:
# Crear dataframe conjunto por dominios usando puntaje ordinal

def _to_ordinal(series):
    mapping = {
        'alteracion_severa': 1,
        'bajo': 2,
        'promedio': 3,
        'alto': 4
    }
    return series.map(mapping)


def _domain_score(df, cols):
    cols_present = [c for c in cols if c in df.columns]
    if not cols_present:
        return pd.Series([None] * len(df), index=df.index)
    scores = df[cols_present].apply(_to_ordinal)
    return scores.mean(axis=1)


# Unir las dos tablas imputadas
cols_id = ['sheet_name', 'nivel_estudio', 'dc', 'age']
df_union = pd.concat([df_tabla_0_imp, df_tabla_1_imp], ignore_index=True, sort=False)

# Definicion de dominios (basado en features)
dominios = {
    'orientacion': ['tiempo', 'persona', 'espacio'],
    'atencion': [
        'atencion_sostenida_auditiva',
        'atencion_sostenida_visual',
        'atencion_selectiva_visual',
        'atencion_dividida_visual',
        'digitos_en_progresion',
        'deteccion_visual',
        'series_sucesivas'
    ],
    'lenguaje': [
        'denominacion',
        'comprension_de_ordenes',
        'material_verbal_complejo',
        'semejanzas',
        'comprension_ejecucion_de_ordenes'
    ],
    'memoria_verbal': [
        'evocacion_inmediata_lista_a',
        'recuerdo_inmediato_lista_a',
        'recuerdo_inmediato_lista_b',
        'recuerdo_libre_a_corto_plazo',
        'recuerdo_libre_a_largo_plazo',
        'reconocimiento',
        'curva_de_memoria_volumen_promedio',
        'memoria_verbal_espontanea_total',
        'memoria_verbal_claves_total',
        'memoria_verbal_reconocimiento',
        'memoria_logica_promedio_historias'
    ],
    'memoria_visual': [
        'evocacion_diferida',
        'caras_codificacion',
        'reconocimiento_caras',
        'evocacion_figura_semicompleja'
    ],
    'gnosias': ['imagenes_sobrepuestas',
                'matrices'],
    'praxis': [
        'constructiva',
        'copia_de_figura_compleja',
        'gestos_simbolicos',
        'imitacion_de_posturas'
    ],
    'ejecutivas': [
        'memoria_de_trabajo_digitos_inversos',
        'memoria_de_trabajo_digitos_secuenciales',
        'evocacion_categorial_semantica',
        'matrices',
        'comprension_abstraccion',
        'stroop_palabra',
        'stroop_colores',
        'stroop_interferencia',
        'fluidez_verbal_semantica',
        'fluidez_verbal_fonologica',
        'fluidez_no_verbal',
        'retencion_digitos_regresion'
    ]
}

# Construir dataframe por dominios
salida = df_union[cols_id].copy()
for dominio, cols in dominios.items():
    salida[dominio] = _domain_score(df_union, cols)

salida.head()

,sheet_name,nivel_estudio,dc,age,orientacion,atencion,lenguaje,memoria_verbal,memoria_visual,gnosias,praxis,ejecutivas
0,gc1-60,0,0,60,4.0,3.25,3.75,2.666667,3.0,NaN,NaN,3.200000
1,gc2-62,0,0,62,4.0,3.00,3.75,2.500000,3.0,3.5,NaN,3.000000
2,f021-72,0,2,72,4.0,2.50,4.00,2.166667,2.0,4.0,3.0,1.500000
3,s2-f067-53,0,1,53,4.0,2.00,3.00,2.000000,3.0,4.0,NaN,2.166667
4,gc3-60,0,0,60,4.0,3.00,3.50,2.666667,3.0,3.5,NaN,3.142857


In [37]:
df_tabla_0[df_tabla_0['sheet_name'] == 's4-f067-66']

,sheet_name,nivel_estudio,dc,age,tiempo,persona,espacio,atencion_sostenida_auditiva,atencion_sostenida_visual,atencion_selectiva_visual,atencion_dividida_visual,denominacion,material_verbal_complejo,comprension_de_ordenes,evocacion_inmediata_lista_a,recuerdo_inmediato_lista_a,recuerdo_inmediato_lista_b,recuerdo_libre_a_corto_plazo,recuerdo_libre_a_largo_plazo,reconocimiento,evocacion_diferida,imagenes_sobrepuestas,matrices,constructiva,memoria_de_trabajo_digitos_inversos,memoria_de_trabajo_digitos_secuenciales,fluidez_verbal_semantica,semejanzas,comprension_abstraccion,stroop_palabra,stroop_color,stroop_interferencia
8,s4-f067-66,0,1,66,alto,alto,alto,promedio,promedio,promedio,promedio,alto,alto,alto,None,None,None,None,None,None,None,None,None,None,promedio,promedio,None,alto,alto,bajo,promedio,bajo


In [38]:
print(salida[salida['gnosias'].isna()][['sheet_name', 'gnosias']])

     sheet_name  gnosias
0        gc1-60      NaN
8    s4-f067-66      NaN
13  s16-f067-60      NaN
15  s34-f067-71      NaN
20  s46-f067-66      NaN
21      gc21-64      NaN
35  s57-f067-55      NaN


In [39]:
salida.isnull().sum()

,0
sheet_name,0
nivel_estudio,0
dc,0
age,0
orientacion,0
atencion,0
lenguaje,0
memoria_verbal,0
memoria_visual,0
gnosias,7


In [40]:
# Patron de nulos: resumen, por grupo clinico y co-ocurrencia

def patron_nulos(df, id_cols, group_col='dc'):
    cols = [c for c in df.columns if c not in id_cols]
    miss = df[cols].isna().astype(int)

    print('--- Resumen global ---')
    resumen = miss.mean().sort_values(ascending=False)
    print(resumen[resumen > 0])

    print('\n--- Nulos por grupo clinico (dc) ---')
    grupo = miss.groupby(df[group_col]).mean().sort_index()
    print(grupo)

    print('\n--- Nulos por paciente (distribucion) ---')
    nulos_paciente = miss.sum(axis=1)
    print(nulos_paciente.describe())

    print('\n--- Co-ocurrencia de nulos (top 10 pares) ---')
    corr = miss.corr()
    corr.values[[range(corr.shape[0])], [range(corr.shape[0])]] = 0
    pares = (
        corr.stack()
        .sort_values(ascending=False)
        .reset_index()
        .rename(columns={'level_0': 'var1', 'level_1': 'var2', 0: 'corr_missing'})
    )
    print(pares.head(10))


print('=== PATRON DE NULOS TABLA 0 ===')
patron_nulos(df_tabla_0, ID_COLS)

print('\n=== PATRON DE NULOS TABLA 1 ===')
patron_nulos(df_tabla_1, ID_COLS)


=== PATRON DE NULOS TABLA 0 ===
--- Resumen global ---
constructiva                               0.972222
fluidez_verbal_semantica                   0.638889
matrices                                   0.333333
imagenes_sobrepuestas                      0.305556
stroop_interferencia                       0.250000
material_verbal_complejo                   0.166667
denominacion                               0.138889
stroop_palabra                             0.111111
evocacion_diferida                         0.111111
stroop_color                               0.111111
atencion_sostenida_visual                  0.111111
memoria_de_trabajo_digitos_secuenciales    0.083333
comprension_abstraccion                    0.083333
semejanzas                                 0.083333
comprension_de_ordenes                     0.055556
reconocimiento                             0.055556
recuerdo_inmediato_lista_b                 0.055556
recuerdo_libre_a_corto_plazo               0.055556
recuerdo_

In [41]:
# Pipeline de entrenamiento con y sin indicadores de ausencia
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier


def preparar_xy(df, target='dc', id_cols=None, drop_cols=None):
    if id_cols is None:
        id_cols = []
    if drop_cols is None:
        drop_cols = []

    X = df.drop(columns=[c for c in [target] + id_cols if c in df.columns])
    if drop_cols:
        X = X.drop(columns=[c for c in drop_cols if c in X.columns])
    y = df[target]
    return X, y


def crear_pipeline(X):
    num_cols = X.select_dtypes(exclude=['object']).columns
    cat_cols = X.select_dtypes(include=['object']).columns

    pre = ColumnTransformer(
        transformers=[
            ('num', 'passthrough', num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
        ],
        remainder='drop'
    )

    model = RandomForestClassifier(
        n_estimators=400,
        random_state=42,
        class_weight='balanced'
    )

    return Pipeline(steps=[('preprocess', pre), ('model', model)])


def evaluar_modelo(df, nombre):
    id_cols = ['age', 'nivel_estudio', 'sheet_name']
    missing_cols = [c for c in df.columns if c.endswith('_missing')]

    print(f'=== {nombre} | Con indicadores de nulos ===')
    X, y = preparar_xy(df, id_cols=id_cols)
    pipe = crear_pipeline(X)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_validate(pipe, X, y, cv=cv, scoring=['accuracy', 'f1_macro'])
    print('CV accuracy:', scores['test_accuracy'].mean().round(3))
    print('CV f1_macro:', scores['test_f1_macro'].mean().round(3))

    # Holdout para reporte detallado
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    print(classification_report(y_test, preds))

    print(f'=== {nombre} | Sin indicadores de nulos ===')
    X2, y2 = preparar_xy(df, id_cols=id_cols, drop_cols=missing_cols)
    pipe2 = crear_pipeline(X2)

    scores2 = cross_validate(pipe2, X2, y2, cv=cv, scoring=['accuracy', 'f1_macro'])
    print('CV accuracy:', scores2['test_accuracy'].mean().round(3))
    print('CV f1_macro:', scores2['test_f1_macro'].mean().round(3))


# Ejecutar para ambas tablas
# Usa las tablas imputadas con indicadores
# df_tabla_0_imp, df_tabla_1_imp

evaluar_modelo(df_tabla_0_imp, 'Tabla 0')
print('\n')
evaluar_modelo(df_tabla_1_imp, 'Tabla 1')


=== Tabla 0 | Con indicadores de nulos ===
CV accuracy: 0.771
CV f1_macro: 0.607
              precision    recall  f1-score   support

           0       0.80      1.00      0.89         4
           1       0.67      0.67      0.67         3
           2       0.00      0.00      0.00         1

    accuracy                           0.75         8
   macro avg       0.49      0.56      0.52         8
weighted avg       0.65      0.75      0.69         8

=== Tabla 0 | Sin indicadores de nulos ===
CV accuracy: 0.771
CV f1_macro: 0.607


=== Tabla 1 | Con indicadores de nulos ===
CV accuracy: 0.79
CV f1_macro: 0.718
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.73      0.89      0.80         9
           2       1.00      0.83      0.91         6

    accuracy                           0.76        17
   macro avg       0.58      0.57      0.57        17
weighted avg       0.74      0.76      0.74      

# Modelos con SMOTE y metricas clinicas

Se entrena con SMOTE para manejar desbalance. Se reportan metricas clinicas por clase: sensibilidad, especificidad, VPP, VPN y F1. Si una clase tiene muy pocos casos, SMOTE puede no aplicarse; en ese caso se informa y se entrena sin SMOTE para esa tabla.

In [42]:
# Entrenamiento con SMOTE para 3 modelos y metricas clinicas
from sklearn.metrics import confusion_matrix
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

try:
    from xgboost import XGBClassifier
except Exception:
    XGBClassifier = None

try:
    from lightgbm import LGBMClassifier
except Exception:
    LGBMClassifier = None


def _onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False)


def _build_preprocessor(X):
    num_cols = X.select_dtypes(exclude=['object']).columns
    cat_cols = X.select_dtypes(include=['object']).columns
    return ColumnTransformer(
        transformers=[
            ('num', 'passthrough', num_cols),
            ('cat', _onehot_encoder(), cat_cols),
        ],
        remainder='drop'
    )


def _smote_for_y(y):
    min_class = y.value_counts().min()
    if min_class < 2:
        return None, min_class
    return SMOTE(random_state=42, k_neighbors=1), min_class


def _coerce_target(y, X):
    y_num = pd.to_numeric(y, errors='coerce')
    mask = y_num.notna()
    if not mask.all():
        X = X.loc[mask].copy()
        y_num = y_num.loc[mask]
    return X, y_num.astype(int)


def clinicas_por_clase(y_true, y_pred, labels):
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    rows = []
    for i, label in enumerate(labels):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - (tp + fn + fp)

        sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
        ppv = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        npv = tn / (tn + fn) if (tn + fn) > 0 else 0.0
        f1 = (2 * ppv * sens / (ppv + sens)) if (ppv + sens) > 0 else 0.0

        rows.append({
            'clase': label,
            'sensibilidad': round(sens, 3),
            'especificidad': round(spec, 3),
            'vpp': round(ppv, 3),
            'vpn': round(npv, 3),
            'f1': round(f1, 3),
            'soporte': int(cm[i, :].sum())
        })

    return pd.DataFrame(rows)


def evaluar_modelo_smote(df, nombre_tabla, modelo, nombre_modelo):
    id_cols = ['age', 'nivel_estudio', 'sheet_name']
    X, y = preparar_xy(df, id_cols=id_cols)
    X, y = _coerce_target(y, X)
    labels = sorted(y.unique())

    pre = _build_preprocessor(X)

    # CV con SMOTE solo si hay suficientes casos por clase
    min_class = y.value_counts().min()
    if min_class >= 3:
        smote_cv, _ = _smote_for_y(y)
        cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

        steps = [('preprocess', pre)]
        if smote_cv is not None:
            steps.append(('smote', smote_cv))
        steps.append(('model', modelo))

        pipe = ImbPipeline(steps=steps)
        scores = cross_validate(pipe, X, y, cv=cv, scoring=['accuracy', 'f1_macro'])
        print(f'=== {nombre_tabla} | {nombre_modelo} ===')
        print('CV accuracy:', scores['test_accuracy'].mean().round(3))
        print('CV f1_macro:', scores['test_f1_macro'].mean().round(3))
    else:
        print(f'=== {nombre_tabla} | {nombre_modelo} ===')
        print('CV con SMOTE omitido: clases muy pequenas para CV estable.')

    # Holdout con SMOTE si el entrenamiento lo permite
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=42
    )

    smote_train, _ = _smote_for_y(y_train)
    steps = [('preprocess', pre)]
    if smote_train is not None:
        steps.append(('smote', smote_train))
    else:
        print('SMOTE omitido en holdout: clase con < 2 casos en train.')
    steps.append(('model', modelo))

    pipe = ImbPipeline(steps=steps)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)

    print(classification_report(y_test, pred))
    print(clinicas_por_clase(y_test, pred, labels))


# Definir modelos
rf = RandomForestClassifier(
    n_estimators=400,
    random_state=42,
    class_weight='balanced'
)

xgb = None
if XGBClassifier is not None:
    xgb = XGBClassifier(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective='multi:softprob',
        num_class=3,
        eval_metric='mlogloss',
        random_state=42
    )

lgbm = None
if LGBMClassifier is not None:
    lgbm = LGBMClassifier(
        n_estimators=400,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        class_weight='balanced'
    )


# Ejecutar para ambas tablas
for df_name, df in [('Tabla 0', df_tabla_0_imp), ('Tabla 1', df_tabla_1_imp)]:
    evaluar_modelo_smote(df, df_name, rf, 'RandomForest')
    print('\n')

    if xgb is not None:
        evaluar_modelo_smote(df, df_name, xgb, 'XGBoost')
        print('\n')
    else:
        print(f'XGBoost no disponible en el entorno para {df_name}.')
        print('\n')

    if lgbm is not None:
        evaluar_modelo_smote(df, df_name, lgbm, 'LightGBM')
        print('\n')
    else:
        print(f'LightGBM no disponible en el entorno para {df_name}.')
        print('\n')


=== Tabla 0 | RandomForest ===
CV accuracy: 0.778
CV f1_macro: 0.554
              precision    recall  f1-score   support

           0       0.80      1.00      0.89         4
           1       0.67      0.67      0.67         3
           2       0.00      0.00      0.00         1

    accuracy                           0.75         8
   macro avg       0.49      0.56      0.52         8
weighted avg       0.65      0.75      0.69         8

   clase  sensibilidad  especificidad    vpp    vpn     f1  soporte
0      0         1.000           0.75  0.800  1.000  0.889        4
1      1         0.667           0.80  0.667  0.800  0.667        3
2      2         0.000           1.00  0.000  0.875  0.000        1


=== Tabla 0 | XGBoost ===
CV accuracy: 0.722
CV f1_macro: 0.517
              precision    recall  f1-score   support

           0       0.80      1.00      0.89         4
           1       0.67      0.67      0.67         3
           2       0.00      0.00      0.00      